In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col

In [3]:
# 1. Inicializar la Sesión de Spark
# Asegúrate de tener PySpark instalado y configurado.
spark = SparkSession.builder \
    .appName("Analisis_Clientes_Behavioural") \
    .getOrCreate()


behavioural_df = spark.read.parquet("/home/jovyan/work/data/BEHAVIOURAL", header=True, inferSchema=True)
clientes_df = spark.read.parquet("/home/jovyan/work/data/CLIENTS", header=True, inferSchema=True)

clientes_psdf = clientes_df.pandas_api()
behavioural_psdf = behavioural_df.pandas_api()

/opt/conda/lib/python3.11/site-packages/pyspark/pandas/__init__.py:50: UserWarning: 'PYARROW_IGNORE_TIMEZONE' environment variable was not set. It is required to set this environment variable to '1' in both driver and executor sides if you use pyarrow>=2.0.0. pandas-on-Spark will set it for you but it does not work if there is a Spark context already launched.
  warnings.warn(


### ANALISIS BEHAVIOURAL

In [4]:
behavioural_psdf.head()

,CONTRACT_ID,CLIENT_ID,DATE,CREDICT_CARD_BALANCE,CREDIT_CARD_LIMIT,CREDIT_CARD_DRAWINGS_ATM,CREDIT_CARD_DRAWINGS,CREDIT_CARD_DRAWINGS_POS,CREDIT_CARD_DRAWINGS_OTHER,CREDIT_CARD_PAYMENT,NUMBER_DRAWINGS_ATM,NUMBER_DRAWINGS,NUMBER_INSTALMENTS,CURRENCY
0,ES1821016961u00XXX,ES182147947X,2020-08-22,0.00,2700.0,0.0,0.0,0.0,0.0,0.00,0.0,0,0.0,euros
1,ES1821083922r00XXX,ES182389511E,2018-11-10,1628.90,1620.0,183.6,183.6,0.0,0.0,113.40,2.0,2,22.0,euros
2,ES1821010991u00XXX,ES182423955L,2021-08-29,0.00,10800.0,0.0,0.0,0.0,0.0,0.00,0.0,0,0.0,euros
3,ES1821002071d00XXX,ES182265271Q,2020-09-22,0.00,810.0,0.0,0.0,0.0,0.0,4.37,0.0,0,16.0,euros
4,ES1821074990u00XXX,ES182332566J,2017-05-01,1692.35,2160.0,0.0,0.0,0.0,0.0,110.70,0.0,0,43.0,euros


In [5]:
print("Shape:", behavioural_psdf.shape)
print("\nTipos:\n", behavioural_psdf.dtypes)
print("\nNulos por columna:\n", behavioural_psdf.isna().sum())
display(behavioural_psdf.describe().T)

Shape: (1724854, 14)

Tipos:
 CONTRACT_ID                    object
CLIENT_ID                      object
DATE                           object
CREDICT_CARD_BALANCE          float64
CREDIT_CARD_LIMIT             float64
CREDIT_CARD_DRAWINGS_ATM      float64
CREDIT_CARD_DRAWINGS          float64
CREDIT_CARD_DRAWINGS_POS      float64
CREDIT_CARD_DRAWINGS_OTHER    float64
CREDIT_CARD_PAYMENT           float64
NUMBER_DRAWINGS_ATM           float64
NUMBER_DRAWINGS                 int32
NUMBER_INSTALMENTS            float64
CURRENCY                       object
dtype: object

Nulos por columna:
 CONTRACT_ID                   0
CLIENT_ID                     0
DATE                          0
CREDICT_CARD_BALANCE          0
CREDIT_CARD_LIMIT             0
CREDIT_CARD_DRAWINGS_ATM      0
CREDIT_CARD_DRAWINGS          0
CREDIT_CARD_DRAWINGS_POS      0
CREDIT_CARD_DRAWINGS_OTHER    0
CREDIT_CARD_PAYMENT           0
NUMBER_DRAWINGS_ATM           0
NUMBER_DRAWINGS               0
NUMBER_INSTALMENTS 

,count,mean,std,min,25%,50%,75%,max
CREDICT_CARD_BALANCE,1724854.0,712.708752,1283.782209,-5043.00,0.0,0.00,1089.81,16257.95
CREDIT_CARD_LIMIT,1724854.0,1829.640860,1956.419156,0.00,540.0,1350.00,2160.00,16200.00
CREDIT_CARD_DRAWINGS_ATM,1724854.0,58.588043,308.722615,-81.93,0.0,0.00,0.00,25380.00
CREDIT_CARD_DRAWINGS,1724854.0,89.589185,405.134845,-74.54,0.0,0.00,0.00,25380.00
CREDIT_CARD_DRAWINGS_POS,1724854.0,27.938011,215.944594,0.00,0.0,0.00,0.00,24720.36
CREDIT_CARD_DRAWINGS_OTHER,1724854.0,2.873773,89.926143,0.00,0.0,0.00,0.00,10800.00
CREDIT_CARD_PAYMENT,1724854.0,99.088714,384.325330,0.00,0.0,5.53,97.20,29237.94
NUMBER_DRAWINGS_ATM,1724854.0,0.255811,1.011515,0.00,0.0,0.00,0.00,44.00
NUMBER_DRAWINGS,1724854.0,0.703787,3.202438,0.00,0.0,0.00,0.00,165.00
NUMBER_INSTALMENTS,1724854.0,19.434148,20.065741,0.00,2.0,13.00,31.00,119.00


In [6]:
missing_values_beh = behavioural_psdf.isnull().sum()

# Filtrar solo las columnas que tienen valores nulos
missing_columns_beh = missing_values_beh[missing_values_beh > 0]

# Mostrar resultados
if missing_columns_beh.empty:
    print("No hay valores nulos en el dataset.")
else:
    print("Valores nulos por columna:")
    for column, missing in missing_columns_beh.items():
        print(f"{column}: {missing} valores nulos")

No hay valores nulos en el dataset.


### ANALISIS CLIENTS

In [7]:
clientes_psdf.head()

,CLIENT_ID,NON_COMPLIANT_CONTRACT,NAME_PRODUCT_TYPE,GENDER,TOTAL_INCOME,AMOUNT_PRODUCT,INSTALLMENT,EDUCATION,MARITAL_STATUS,HOME_SITUATION,REGION_SCORE,AGE_IN_YEARS,JOB_SENIORITY,HOME_SENIORITY,LAST_UPDATE,OWN_INSURANCE_CAR,CAR_AGE,FAMILY_SIZE,REACTIVE_SCORING,PROACTIVE_SCORING,BEHAVIORAL_SCORING,DAYS_LAST_INFO_CHANGE,NUMBER_OF_PRODUCTS,OCCUPATION,DIGITAL_CLIENT,HOME_OWNER,EMPLOYER_ORGANIZATION_TYPE,CURRENCY,NUM_PREVIOUS_LOAN_APP,LOAN_ANNUITY_PAYMENT_MAX,LOAN_ANNUITY_PAYMENT_MIN,LOAN_ANNUITY_PAYMENT_SUM,LOAN_APPLICATION_AMOUNT_MAX,LOAN_APPLICATION_AMOUNT_MIN,LOAN_APPLICATION_AMOUNT_SUM,LOAN_CREDIT_GRANTED_MAX,LOAN_CREDIT_GRANTED_MIN,LOAN_CREDIT_GRANTED_SUM,LOAN_VARIABLE_RATE_MAX,LOAN_VARIABLE_RATE_MIN,NUM_STATUS_ANNULLED,NUM_STATUS_AUTHORIZED,NUM_STATUS_DENIED,NUM_STATUS_NOT_USED,NUM_FLAG_INSURED
0,ES182430463G,0,PRODUCT 1,F,1458.0,9062.28,337.39,Secondary,Single,House,0.032561,42.835616,853.0,9771.0,4509.0,N,NaN,3.0,NaN,0.515971,0.488455,5.0,0.0,Payroll,0,N,0006,euros,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,ES182321930B,0,PRODUCT 1,F,1620.0,9046.08,240.19,Secondary,Single,House,0.020246,47.460274,9587.0,5732.0,873.0,N,NaN,1.0,0.486148,0.384908,0.187389,2815.0,3.0,Payroll,1,Y,0001,euros,18.0,354.68,0.00,1965.98,8100.0,0.00,34691.22,10731.69,0.00,43653.38,0.141975,0.0,4.0,6.0,8.0,0.0,3.0
2,ES182200938W,0,PRODUCT 1,M,3780.0,8154.00,417.91,None,Married,House,0.022800,31.961644,1904.0,9.0,4154.0,Y,1.0,3.0,0.038827,0.624212,0.638044,1347.0,3.0,Payroll,1,Y,0018,euros,11.0,439.31,72.28,2490.25,5400.0,581.63,22970.54,6683.42,447.28,24686.48,0.289639,0.0,0.0,8.0,3.0,0.0,2.0
3,ES182445305J,0,PRODUCT 1,F,2160.0,15460.20,453.60,Secondary,Married,House,0.011703,36.736986,3744.0,9700.0,3945.0,N,NaN,2.0,NaN,0.753599,0.629674,527.0,2.0,Payroll,0,Y,0010,euros,5.0,135.00,27.00,357.50,2700.0,328.21,5616.97,2700.00,328.21,5412.10,0.108909,0.0,0.0,4.0,1.0,0.0,0.0
4,ES182173856D,1,PRODUCT 1,M,1890.0,4206.98,453.06,None,Single,House,0.015221,30.649315,1693.0,3772.0,3857.0,N,NaN,1.0,0.196233,0.605344,0.411849,119.0,7.0,Payroll,0,Y,0010,euros,13.0,149.30,0.00,544.79,2160.0,0.00,7371.60,2160.00,0.00,7982.02,0.189254,0.0,6.0,4.0,2.0,1.0,1.0


In [22]:
print("Shape:", clientes_psdf.shape)
print("\nTipos:\n", clientes_psdf.dtypes)
print("\nNulos por columna:\n", clientes_psdf.isna().sum())
display(clientes_psdf.describe().T)

Shape: (162977, 45)

Tipos:
 CLIENT_ID                       object
NON_COMPLIANT_CONTRACT           int32
NAME_PRODUCT_TYPE               object
GENDER                          object
TOTAL_INCOME                   float64
AMOUNT_PRODUCT                 float64
INSTALLMENT                    float64
EDUCATION                       object
MARITAL_STATUS                  object
HOME_SITUATION                  object
REGION_SCORE                   float64
AGE_IN_YEARS                   float64
JOB_SENIORITY                  float64
HOME_SENIORITY                 float64
LAST_UPDATE                    float64
OWN_INSURANCE_CAR               object
CAR_AGE                        float64
FAMILY_SIZE                    float64
REACTIVE_SCORING               float64
PROACTIVE_SCORING              float64
BEHAVIORAL_SCORING             float64
DAYS_LAST_INFO_CHANGE          float64
NUMBER_OF_PRODUCTS             float64
OCCUPATION                      object
DIGITAL_CLIENT                   in

,count,mean,std,min,25%,50%,75%,max
NON_COMPLIANT_CONTRACT,162977.0,0.081214,0.273164,0.000000,0.000000,0.000000,0.000000,1.000000e+00
TOTAL_INCOME,162977.0,2029.338959,3722.499128,307.800000,1350.000000,1782.000000,2430.000000,1.404000e+06
AMOUNT_PRODUCT,162977.0,7193.391443,4831.534717,540.000000,3240.000000,6162.370000,9703.800000,4.848619e+04
INSTALLMENT,162970.0,325.583354,173.477501,19.390000,198.880000,299.160000,415.640000,2.898155e+03
REGION_SCORE,162977.0,0.020516,0.012600,0.000533,0.010006,0.018850,0.028663,5.936400e-02
AGE_IN_YEARS,162977.0,43.951931,11.931164,20.517808,34.073973,43.197260,53.879452,6.908219e+01
JOB_SENIORITY,133803.0,2398.921459,2360.495553,1.000000,770.000000,1650.000000,3193.000000,1.772900e+04
HOME_SENIORITY,162977.0,4987.488180,3520.915689,0.000000,2010.000000,4507.000000,7482.000000,2.404400e+04
LAST_UPDATE,162977.0,2992.176141,1510.215965,0.000000,1713.000000,3254.000000,4297.000000,6.874000e+03
CAR_AGE,55427.0,11.987407,11.799483,0.000000,5.000000,9.000000,15.000000,6.450000e+01


In [23]:
missing_values_cli = clientes_psdf.isnull().sum()

# Filtrar solo las columnas que tienen valores nulos
missing_columns_cli = missing_values_cli[missing_values_cli > 0]

# Mostrar resultados
if missing_columns_cli.empty:
    print("No hay valores nulos en el dataset.")
else:
    print("Valores nulos por columna:")
    for column, missing in missing_columns_cli.items():
        print(f"{column}: {missing} valores nulos")

Valores nulos por columna:
INSTALLMENT: 7 valores nulos
EDUCATION: 39640 valores nulos
MARITAL_STATUS: 2 valores nulos
JOB_SENIORITY: 29174 valores nulos
CAR_AGE: 107550 valores nulos
FAMILY_SIZE: 2 valores nulos
REACTIVE_SCORING: 91901 valores nulos
PROACTIVE_SCORING: 337 valores nulos
BEHAVIORAL_SCORING: 32246 valores nulos
DAYS_LAST_INFO_CHANGE: 1 valores nulos
NUMBER_OF_PRODUCTS: 21903 valores nulos
EMPLOYER_ORGANIZATION_TYPE: 29464 valores nulos
NUM_PREVIOUS_LOAN_APP: 8770 valores nulos
LOAN_ANNUITY_PAYMENT_MAX: 8770 valores nulos
LOAN_ANNUITY_PAYMENT_MIN: 8770 valores nulos
LOAN_ANNUITY_PAYMENT_SUM: 8770 valores nulos
LOAN_APPLICATION_AMOUNT_MAX: 8770 valores nulos
LOAN_APPLICATION_AMOUNT_MIN: 8770 valores nulos
LOAN_APPLICATION_AMOUNT_SUM: 8770 valores nulos
LOAN_CREDIT_GRANTED_MAX: 8770 valores nulos
LOAN_CREDIT_GRANTED_MIN: 8770 valores nulos
LOAN_CREDIT_GRANTED_SUM: 8770 valores nulos
LOAN_VARIABLE_RATE_MAX: 8770 valores nulos
LOAN_VARIABLE_RATE_MIN: 8770 valores nulos
NUM_ST

### DATASETS DE ID'S QUE COINCIDEN EN AMBOS DATASETS

In [24]:
df_coincidencias = clientes_df.join(
    behavioural_df,
    on="CLIENT_ID",
    how="inner"
)

clientes_idos = behavioural_df.count() - clientes_df.count()

print(f"Hay {clientes_df.count()} clientes que se han quedado")
print(f"Hay {clientes_idos} clientes se han ido")

print(f"Un {round((clientes_idos/clientes_df.count())*10, 3)}% de los clientes se han ido")




Hay 162977 clientes que se han quedado
Hay 1561877 clientes se han ido
Un 95.834% de los clientes se han ido


### Porcentajes de missing values

In [26]:
n_filas_clientes = clientes_psdf.shape[0]
n_filas_behavioural = behavioural_psdf.shape[0]

#PORCENTAJE DE VALORES NULOS 
if missing_columns_beh.empty:
    print("No hay valores nulos en las variables numéricas.")
else:
    print("Valores nulos en variables numéricas:")
    for column, missing in missing_columns_beh.items():
        porcentaje = (missing / n_filas_behavioural) * 100
        print(f"{column}: {missing} NaN ({porcentaje:.2f}%)")


print("\n\n")

#PORCENTAJE DE VALORES NULOS 
if missing_columns_cli.empty:
    print("No hay valores nulos en las variables numéricas.")
else:
    print("Valores nulos en variables numéricas:")
    for column, missing in missing_columns_cli.items():
        porcentaje = (missing / n_filas_clientes) * 100
        print(f"{column}: {missing} NaN ({porcentaje:.2f}%)")

No hay valores nulos en las variables numéricas.



Valores nulos en variables numéricas:
INSTALLMENT: 7 NaN (0.00%)
EDUCATION: 39640 NaN (24.32%)
MARITAL_STATUS: 2 NaN (0.00%)
JOB_SENIORITY: 29174 NaN (17.90%)
CAR_AGE: 107550 NaN (65.99%)
FAMILY_SIZE: 2 NaN (0.00%)
REACTIVE_SCORING: 91901 NaN (56.39%)
PROACTIVE_SCORING: 337 NaN (0.21%)
BEHAVIORAL_SCORING: 32246 NaN (19.79%)
DAYS_LAST_INFO_CHANGE: 1 NaN (0.00%)
NUMBER_OF_PRODUCTS: 21903 NaN (13.44%)
EMPLOYER_ORGANIZATION_TYPE: 29464 NaN (18.08%)
NUM_PREVIOUS_LOAN_APP: 8770 NaN (5.38%)
LOAN_ANNUITY_PAYMENT_MAX: 8770 NaN (5.38%)
LOAN_ANNUITY_PAYMENT_MIN: 8770 NaN (5.38%)
LOAN_ANNUITY_PAYMENT_SUM: 8770 NaN (5.38%)
LOAN_APPLICATION_AMOUNT_MAX: 8770 NaN (5.38%)
LOAN_APPLICATION_AMOUNT_MIN: 8770 NaN (5.38%)
LOAN_APPLICATION_AMOUNT_SUM: 8770 NaN (5.38%)
LOAN_CREDIT_GRANTED_MAX: 8770 NaN (5.38%)
LOAN_CREDIT_GRANTED_MIN: 8770 NaN (5.38%)
LOAN_CREDIT_GRANTED_SUM: 8770 NaN (5.38%)
LOAN_VARIABLE_RATE_MAX: 8770 NaN (5.38%)
LOAN_VARIABLE_RATE_MIN: 8